WEEK_7
AYUSHI GUPTA

OBJECTIVE:-
To implement incremental data processing using Delta Lake by loading customer data, cleaning it, creating a Delta table, merging incremental updates, and validating the final dataset.

In [0]:
# ======================================================
# Delta Lake Assignment
# Step 1 : Load Required Libraries
# ======================================================
from pyspark.sql.functions import *
from delta.tables import *

In [0]:
# ======================================================
# Step 2 : Load Customer Master Dataset
# ======================================================

master_df = spark.read.csv(
"/Volumes/workspace/default/assignment_data/customer_master.csv",
header=True,
inferSchema=True
)

display(master_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email
1001,Customer_1001,Mumbai,Electronics,2353.0,4,5,customer1001@example.com
1002,Customer_1002,Jaipur,Electronics,4567.0,2,15,customer1002@example.com
1003,Customer_1003,Delhi,Electronics,867.0,4,5,customer1003@example.com
1004,Customer_1004,Lucknow,Sports,317.0,9,5,customer1004@example.com
1005,Customer_1005,Lucknow,Grocery,1905.0,8,10,customer1005@example.com
1006,Customer_1006,null,Furniture,3562.0,6,10,customer1006@example.com
1007,Customer_1007,Jaipur,Furniture,2857.0,2,0,customer1007@example.com
1008,Customer_1008,Chennai,Electronics,3040.0,6,10,customer1008@example.com
1009,Customer_1009,Delhi,Grocery,4492.0,2,15,customer1009@example.com
1010,Customer_1010,Mumbai,Sports,2501.0,10,10,customer1010@example.com


In [0]:
# Check column names and data types
master_df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Email: string (nullable = true)



In [0]:
# Total number of rows
print("Total Rows:", master_df.count())

Total Rows: 103


In [0]:
from pyspark.sql.functions import col, count, when

master_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in master_df.columns
]).show()

+----------+------------+----+--------+-----+--------+--------+-----+
|CustomerID|CustomerName|City|Category|Price|Quantity|Discount|Email|
+----------+------------+----+--------+-----+--------+--------+-----+
|         0|           0|   4|       0|    2|       0|       0|    0|
+----------+------------+----+--------+-----+--------+--------+-----+



In [0]:
# Remove rows containing null values
master_df = master_df.dropna()

display(master_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email
1001,Customer_1001,Mumbai,Electronics,2353.0,4,5,customer1001@example.com
1002,Customer_1002,Jaipur,Electronics,4567.0,2,15,customer1002@example.com
1003,Customer_1003,Delhi,Electronics,867.0,4,5,customer1003@example.com
1004,Customer_1004,Lucknow,Sports,317.0,9,5,customer1004@example.com
1005,Customer_1005,Lucknow,Grocery,1905.0,8,10,customer1005@example.com
1007,Customer_1007,Jaipur,Furniture,2857.0,2,0,customer1007@example.com
1008,Customer_1008,Chennai,Electronics,3040.0,6,10,customer1008@example.com
1009,Customer_1009,Delhi,Grocery,4492.0,2,15,customer1009@example.com
1010,Customer_1010,Mumbai,Sports,2501.0,10,10,customer1010@example.com
1011,Customer_1011,Indore,Furniture,669.0,1,5,customer1011@example.com


In [0]:
# Remove duplicate records
master_df = master_df.dropDuplicates()

display(master_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email
1007,Customer_1007,Jaipur,Furniture,2857.0,2,0,customer1007@example.com
1011,Customer_1011,Indore,Furniture,669.0,1,5,customer1011@example.com
1021,Customer_1021,Indore,Grocery,4880.0,7,10,customer1021@example.com
1086,Customer_1086,Indore,Grocery,2931.0,6,15,customer1086@example.com
1090,Customer_1090,Mumbai,Clothing,4254.0,5,15,customer1090@example.com
1020,Customer_1020,Jaipur,Furniture,4698.0,9,10,customer1020@example.com
1051,Customer_1051,Chennai,Electronics,1448.0,7,0,customer1051@example.com
1058,Customer_1058,Chennai,Electronics,4766.0,4,0,customer1058@example.com
1062,Customer_1062,Mumbai,Electronics,3854.0,10,0,customer1062@example.com
1042,Customer_1042,Indore,Sports,3972.0,4,15,customer1042@example.com


In [0]:
# Save cleaned data as a Delta table
master_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/assignment_data/customer_delta")
delta_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/assignment_data/customer_delta"
)

display(delta_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email
1007,Customer_1007,Jaipur,Furniture,2857.0,2,0,customer1007@example.com
1011,Customer_1011,Indore,Furniture,669.0,1,5,customer1011@example.com
1021,Customer_1021,Indore,Grocery,4880.0,7,10,customer1021@example.com
1086,Customer_1086,Indore,Grocery,2931.0,6,15,customer1086@example.com
1090,Customer_1090,Mumbai,Clothing,4254.0,5,15,customer1090@example.com
1020,Customer_1020,Jaipur,Furniture,4698.0,9,10,customer1020@example.com
1051,Customer_1051,Chennai,Electronics,1448.0,7,0,customer1051@example.com
1058,Customer_1058,Chennai,Electronics,4766.0,4,0,customer1058@example.com
1062,Customer_1062,Mumbai,Electronics,3854.0,10,0,customer1062@example.com
1042,Customer_1042,Indore,Sports,3972.0,4,15,customer1042@example.com


In [0]:
incremental_df = spark.read.csv(
    "/Volumes/workspace/default/assignment_data/customer_incremental.csv",
    header=True,
    inferSchema=True
)

display(incremental_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email
1021,Customer_1021,Mumbai,Grocery,5265.0,1,10,customer1021@example.com
1022,Customer_1022,Pune,Furniture,4722.0,7,0,customer1022@example.com
1023,Customer_1023,Kolkata,Electronics,1492.0,9,15,null
1024,Customer_1024,Pune,Electronics,3327.0,9,15,customer1024@example.com
1025,Customer_1025,Kolkata,Clothing,5116.0,2,0,customer1025@example.com
1026,Customer_1026,Lucknow,Clothing,3004.0,8,10,customer1026@example.com
1027,Customer_1027,Hyderabad,Furniture,4152.0,10,10,customer1027@example.com
1028,Customer_1028,Indore,Furniture,4694.0,8,10,customer1028@example.com
1029,Customer_1029,Chennai,Sports,2196.0,9,10,customer1029@example.com
1030,Customer_1030,Kolkata,Sports,4919.0,9,10,customer1030@example.com


In [0]:
incremental_df.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Email: string (nullable = true)



In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/default/assignment_data/customer_delta"
)

In [0]:
incremental_df.groupBy("CustomerID") \
    .count() \
    .filter("count > 1") \
    .show()

+----------+-----+
|CustomerID|count|
+----------+-----+
|      1026|    2|
+----------+-----+



In [0]:
# Remove duplicate CustomerIDs
incremental_df = incremental_df.dropDuplicates(["CustomerID"])

display(incremental_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email
1021,Customer_1021,Mumbai,Grocery,5265.0,1,10,customer1021@example.com
1030,Customer_1030,Kolkata,Sports,4919.0,9,10,customer1030@example.com
1044,Customer_1044,Hyderabad,Grocery,3467.0,8,0,customer1044@example.com
1108,Customer_1108,Indore,Sports,322.0,2,15,customer1108@example.com
1022,Customer_1022,Pune,Furniture,4722.0,7,0,customer1022@example.com
1025,Customer_1025,Kolkata,Clothing,5116.0,2,0,customer1025@example.com
1060,Customer_1060,Bengaluru,Furniture,2673.0,4,10,customer1060@example.com
1122,Customer_1122,Delhi,Grocery,4115.0,2,15,customer1122@example.com
1026,Customer_1026,Lucknow,Clothing,3004.0,8,10,customer1026@example.com
1041,Customer_1041,Bengaluru,Grocery,1855.0,9,5,customer1041@example.com


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(
    spark,
    "/Volumes/workspace/default/assignment_data/customer_delta"
)

deltaTable.alias("old") \
.merge(
    incremental_df.alias("new"),
    "old.CustomerID = new.CustomerID"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
final_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/assignment_data/customer_delta"
)

display(final_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email
1007,Customer_1007,Jaipur,Furniture,2857.0,2,0,customer1007@example.com
1011,Customer_1011,Indore,Furniture,669.0,1,5,customer1011@example.com
1086,Customer_1086,Indore,Grocery,2931.0,6,15,customer1086@example.com
1090,Customer_1090,Mumbai,Clothing,4254.0,5,15,customer1090@example.com
1020,Customer_1020,Jaipur,Furniture,4698.0,9,10,customer1020@example.com
1062,Customer_1062,Mumbai,Electronics,3854.0,10,0,customer1062@example.com
1065,Customer_1065,Bengaluru,Furniture,3689.0,9,10,customer1065@example.com
1097,Customer_1097,Mumbai,Furniture,2642.0,4,5,customer1097@example.com
1072,Customer_1072,Jaipur,Grocery,4619.0,7,0,customer1072@example.com
1077,Customer_1077,Lucknow,Grocery,1366.0,4,5,customer1077@example.com


In [0]:
print("Total Rows:", final_df.count())

Total Rows: 126


In [0]:
final_df.groupBy("CustomerID").count().show()

+----------+-----+
|CustomerID|count|
+----------+-----+
|      1088|    1|
|      1092|    1|
|      1003|    1|
|      1021|    1|
|      1030|    1|
|      1044|    1|
|      1108|    1|
|      1084|    1|
|      1094|    1|
|      1100|    1|
|      1015|    1|
|      1089|    1|
|      1022|    1|
|      1025|    1|
|      1060|    1|
|      1122|    1|
|      1007|    1|
|      1087|    1|
|      1076|    1|
|      1002|    1|
+----------+-----+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import *

final_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in final_df.columns
]).show()

+----------+------------+----+--------+-----+--------+--------+-----+
|CustomerID|CustomerName|City|Category|Price|Quantity|Discount|Email|
+----------+------------+----+--------+-----+--------+--------+-----+
|         0|           0|   0|       1|    1|       0|       0|    1|
+----------+------------+----+--------+-----+--------+--------+-----+



In [0]:
final_df = final_df.dropna()

In [0]:
# Save final dataframe as a single CSV file
output_path = "/Volumes/workspace/default/assignment_data/final_output"

final_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("CSV saved successfully!")

CSV saved successfully!


In [0]:
display(final_df)

CustomerID,CustomerName,City,Category,Price,Quantity,Discount,Email
1007,Customer_1007,Jaipur,Furniture,2857.0,2,0,customer1007@example.com
1011,Customer_1011,Indore,Furniture,669.0,1,5,customer1011@example.com
1086,Customer_1086,Indore,Grocery,2931.0,6,15,customer1086@example.com
1090,Customer_1090,Mumbai,Clothing,4254.0,5,15,customer1090@example.com
1020,Customer_1020,Jaipur,Furniture,4698.0,9,10,customer1020@example.com
1062,Customer_1062,Mumbai,Electronics,3854.0,10,0,customer1062@example.com
1065,Customer_1065,Bengaluru,Furniture,3689.0,9,10,customer1065@example.com
1097,Customer_1097,Mumbai,Furniture,2642.0,4,5,customer1097@example.com
1072,Customer_1072,Jaipur,Grocery,4619.0,7,0,customer1072@example.com
1077,Customer_1077,Lucknow,Grocery,1366.0,4,5,customer1077@example.com


In [0]:
final_df.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/assignment_data/final_csv")

In [0]:
print("========== Assignment Summary ==========")
print("Master dataset loaded successfully.")
print("Data cleaning completed (null values and duplicates handled).")
print("Delta table created successfully.")
print("Incremental dataset loaded successfully.")
print("MERGE operation completed successfully.")
print("Final merged dataset validated.")
print("Final record count:", final_df.count())

========== Assignment Summary ==========
Master dataset loaded successfully.
Data cleaning completed (null values and duplicates handled).
Delta table created successfully.
Incremental dataset loaded successfully.
MERGE operation completed successfully.
Final merged dataset validated.
Final record count: 123
